In [0]:
%pip install chromadb

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.7/22.7 MB 157.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 177.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 88.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 67.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 109.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 119.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 41.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 97.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.2/549.2 kB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 153.0 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Not uninstalling urllib3 at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/env

In [0]:
# =====================================================
# TECH CHALLENGE FASE 4
# ETAPA: RETRIEVER + CHROMADB
#
# Objetivo:
# Construir e avaliar o mecanismo de recuperação
# semântica (Retriever) do sistema RAG.
# =====================================================

import time
import pandas as pd
import numpy as np
import chromadb

ETAPA 2 - Conectar ao Banco Vetorial

In [0]:
# =====================================================
# CONEXÃO COM O CHROMADB
# =====================================================

# PersistentClient mantém os vetores salvos.

client = chromadb.PersistentClient(
    path="/tmp/chroma_olist"
)

# Collection equivale a uma tabela.
# ATENCAO: /tmp e apagado quando o compute muda. Se a collection
# nao existir, ela e reconstruida a partir da tabela Gold de
# embeddings ja calculados (projeto.gold.customer_reviews_embeddings),
# sem precisar recalcular os embeddings.

collection = client.get_or_create_collection(
    name="olist_reviews"
)

if collection.count() == 0:
    kb_pd = (
        spark.table("projeto.gold.customer_reviews_embeddings")
        .select("document_id", "document_text", "embedding")
        .toPandas()
    )
    batch = 1000
    total = len(kb_pd)
    for start in range(0, total, batch):
        chunk = kb_pd.iloc[start:start + batch]
        collection.add(
            ids=[str(i) for i in chunk["document_id"]],
            documents=chunk["document_text"].tolist(),
            embeddings=[list(e) for e in chunk["embedding"]],
        )
    print(f"Collection reconstruida: {collection.count()} documentos")
else:
    print(f"Collection encontrada: {collection.count()} documentos")

Collection reconstruida: 48272 documentos


ETAPA 3 - Validar a Base

In [0]:
# =====================================================
# VALIDAÇÃO DA BASE INDEXADA
# =====================================================

total_documentos = collection.count()

print(
    f"Total de documentos indexados: {total_documentos:,}"
)

Total de documentos indexados: 48,272


ETAPA 4 - Função de Busca

In [0]:
# =====================================================
# FUNÇÃO DE RECUPERAÇÃO
# =====================================================

def buscar_reviews(
        pergunta,
        top_k=5):

    """
    Recebe uma pergunta.

    Converte para embedding.

    Recupera os documentos
    semanticamente mais parecidos.
    """

    resultado = collection.query(

        query_texts=[
            pergunta
        ],

        n_results=top_k

    )

    return resultado

ETAPA 5 - Perguntas de Teste

In [0]:
# =====================================================
# CASOS DE TESTE
# =====================================================

perguntas_teste = [

    "atraso na entrega",

    "produto quebrado",

    "demora para chegar",

    "qualidade do produto",

    "atendimento ruim",

    "problemas de transporte",

    "embalagem danificada",

    "troca de produto"

]

ETAPA 6 - Visualizar Recuperação

In [0]:
# =====================================================
# EXIBIÇÃO DOS DOCUMENTOS RECUPERADOS
# =====================================================

for pergunta in perguntas_teste:

    print("\n")
    print("=" * 100)

    print(
        f"PERGUNTA: {pergunta}"
    )

    print("=" * 100)

    resultado = buscar_reviews(
        pergunta,
        top_k=3
    )

    documentos = resultado["documents"][0]

    metadados = resultado["metadatas"][0]

    distancias = resultado["distances"][0]

    for posicao in range(
        len(documentos)
    ):

        print("\n")

        print(
            f"TOP {posicao + 1}"
        )

        print("-" * 50)

        print(
            "DOCUMENTO:"
        )

        print(
            documentos[posicao]
        )

        print()

        print(
            "METADADOS:"
        )

        print(
            metadados[posicao]
        )

        print()

        print(
            "DISTÂNCIA:"
        )

        print(
            round(
                distancias[posicao],
                4
            )
        )




PERGUNTA: atraso na entrega


/home/spark-a770f062-3b70-4d4f-8a73-6e/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s]




TOP 1
--------------------------------------------------
DOCUMENTO:
Categoria do Produto: informatica_acessorios Nota da Avaliação: 4 Sentimento: POSITIVO Comentário do Cliente: RECOMENDO A LOJA E O PRODUTO, SÓ NÃO É PEFEITO PORQUE OS CORREIOS NÃO ENTREGAM ONDE MORO POR CAUSA DOS ASSALTOS. MAS O RESTENTE É PERFEITO, ALOJA, O PRODUTO O ATENDIMENTO, O PRAZO. OBRIGADA, VOLTAREI

METADADOS:
None

DISTÂNCIA:
3.1334


TOP 2
--------------------------------------------------
DOCUMENTO:
Categoria do Produto: cama_mesa_banho Nota da Avaliação: 2 Sentimento: NEGATIVO Comentário do Cliente: Um verdadeiro absurdo hj já são 09 de agosto e nada da entrega do meu produto. Uma verdadeira falta de respeito. O stark deveria escolher melhor o seu meio de transpirte

METADADOS:
None

DISTÂNCIA:
3.3977


TOP 3
--------------------------------------------------
DOCUMENTO:
Categoria do Produto: ferramentas_jardim Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Continuo aguardando o produto

ETAPA 7 - Tempo Médio de Recuperação

In [0]:
# =====================================================
# TEMPO DE RECUPERAÇÃO
# =====================================================

tempos = []

for pergunta in perguntas_teste:

    inicio = time.time()

    buscar_reviews(
        pergunta,
        5
    )

    fim = time.time()

    tempos.append(
        fim - inicio
    )

tempo_medio = np.mean(
    tempos
)

print(
    f"Tempo médio: {tempo_medio:.4f} segundos"
)

Tempo médio: 0.2927 segundos


ETAPA 8 - Distância Média
Mede similaridade.

Quanto menor, melhor.

In [0]:
# =====================================================
# DISTÂNCIA MÉDIA
# =====================================================

distancias_todas = []

for pergunta in perguntas_teste:

    resultado = buscar_reviews(
        pergunta,
        5
    )

    distancias_todas.extend(
        resultado["distances"][0]
    )

distancia_media = np.mean(
    distancias_todas
)

print(
    f"Distância média: {distancia_media:.4f}"
)

Distância média: 3.2378


ETAPA 9 - Distribuição dos Sentimentos

In [0]:
# =====================================================
# DISTRIBUIÇÃO DOS SENTIMENTOS
# =====================================================

# A collection não armazena metadados. O sentimento é
# recuperado da tabela Gold pelo document_id de cada
# documento retornado.

sentimento_map = (

    spark
    .table(
        "projeto.gold."
        "customer_reviews_knowledge_base"
    )

    .select(
        "document_id",
        "sentimento"
    )

    .toPandas()

    .set_index("document_id")
    ["sentimento"]

    .to_dict()

)

sentimentos = []

for pergunta in perguntas_teste:

    resultado = buscar_reviews(
        pergunta,
        10
    )

    for doc_id in resultado[
        "ids"
    ][0]:

        sentimentos.append(
            sentimento_map[
                int(doc_id)
            ]
        )

sentimentos_df = pd.DataFrame(
    sentimentos,
    columns=["sentimento"]
)

display(

    sentimentos_df
    .groupby(
        "sentimento"
    )
    .size()
    .reset_index(
        name="quantidade"
    )

)

sentimento,quantidade
NEGATIVO,40
NEUTRO,13
POSITIVO,27


ETAPA 10 - Tabela de Avaliação

In [0]:
# =====================================================
# AVALIAÇÃO FINAL
# =====================================================

metricas = {

    "documentos_indexados":
        collection.count(),

    "embedding_model":
        "all-MiniLM-L6-v2",

    "dimensao_embedding":
        384,

    "tempo_medio_busca":
        round(
            tempo_medio,
            4
        ),

    "distancia_media":
        round(
            distancia_media,
            4
        ),

    "top_k":
        5

}

ETAPA 11 - Exibir Métricas

In [0]:
# =====================================================
# DASHBOARD DE MÉTRICAS
# =====================================================

print("\n")

print("=" * 60)

print(
    "MÉTRICAS DO RETRIEVER"
)

print("=" * 60)

for k,v in metricas.items():

    print(
        f"{k}: {v}"
    )



MÉTRICAS DO RETRIEVER
documentos_indexados: 48272
embedding_model: all-MiniLM-L6-v2
dimensao_embedding: 384
tempo_medio_busca: 0.2927
distancia_media: 3.2378
top_k: 5


Experimento B
paraphrase-multilingual-MiniLM-L12-v2
+
ChromaDB

In [0]:
CONFIG = {

    "TOP_K": 5,

    "BATCH_SIZE": 500,

    "CHROMA_PATH":
    "/tmp/chroma_olist",

    "MODELS": [

        "sentence-transformers/all-MiniLM-L6-v2",

        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

    ]

}

SEÇÃO 2 - CARGA

In [0]:
from pyspark.sql import functions as F

kb = (

    spark.table(
        "gold.customer_reviews_knowledge_base"
    )

    .filter(
        F.length(
            "document_text"
        ) > 20
    )

    .dropDuplicates(
        ["document_text"]
    )

)

Reviews vazios pioram embeddings.
Reviews duplicados aumentam ruído.
SEÇÃO 3 - PREPARAÇÃO

In [0]:
from pyspark.sql import functions as F

kb_pd = (

    spark.table(
        "projeto.gold."
        "customer_reviews_knowledge_base"
    )

    .filter(
        F.length(
            "document_text"
        ) > 20
    )

    .dropDuplicates(
        ["document_text"]
    )

    .select(

        "document_id",

        "document_text",

        "review_score",

        "sentimento",

        "product_category_name",

        "dias_entrega"

    ).toPandas()

)

SEÇÃO 4 - EMBEDDINGS

Criar função única.

In [0]:
%pip install sentence-transformers

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 19.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 136.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.9/804.9 kB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 483.7/483.7 MB 79.1 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.3/595.3 MB 73.8 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.1/221.1 MB 95.7 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 71.9 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 93.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 62.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 MB 78.5 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 81.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 MB 73.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
from sentence_transformers import SentenceTransformer

def gerar_embeddings(
        modelo,
        textos):

    model = SentenceTransformer(
        modelo
    )

    embeddings = model.encode(

        textos,

        batch_size=128,

        show_progress_bar=True

    )

    return embeddings

/local_disk0/.ephemeral_nfs/envs/pythonEnv-e9174953-5e22-41d1-af44-487fc702505e/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


Otimização 1

Persistir embeddings.

Não regenerar toda vez.

SEÇÃO 5 - CHROMADB

Função genérica.

In [0]:
import chromadb

def criar_chroma():

    client = chromadb.PersistentClient(
        path="/tmp/chroma_olist"
    )

    return client

In [0]:
batch_size = 500

Menor uso de memória.

Menor risco de timeout.

SEÇÃO 6 - RETRIEVER

In [0]:
def buscar(

    collection,

    pergunta,

    top_k=5

):

    return collection.query(

        query_texts=[pergunta],

        n_results=top_k

    )

SEÇÃO 7 - AVALIAÇÃO

In [0]:
PERGUNTAS = [

    "atraso na entrega",

    "produto quebrado",

    "problemas de transporte",

    "demora para chegar",

    "qualidade do produto",

    "atendimento ruim",

    "embalagem danificada",

    "cancelamento"

]

Desvio padrão.

In [0]:
import numpy as np
import chromadb

# Garante a collection populada. O /tmp e apagado quando o
# compute muda: se estiver vazia, recarrega os embeddings ja
# calculados (paraphrase-multilingual-MiniLM-L12-v2) da tabela Gold.

client = chromadb.PersistentClient(
    path="/tmp/chroma_olist"
)

collection = client.get_or_create_collection(
    name="olist_reviews"
)

if collection.count() == 0:

    kb_emb = (
        spark.table(
            "projeto.gold."
            "customer_reviews_embeddings"
        )
        .select(
            "document_id",
            "document_text",
            "embedding"
        )
        .toPandas()
    )

    batch = 500

    for start in range(0, len(kb_emb), batch):

        chunk = kb_emb.iloc[start:start + batch]

        collection.add(
            ids=[str(i) for i in chunk["document_id"]],
            documents=chunk["document_text"].tolist(),
            embeddings=[list(e) for e in chunk["embedding"]],
        )

# Distancias de recuperacao dos top-5 por pergunta

distancias = []

for pergunta in PERGUNTAS:

    resultado = buscar(
        collection,
        pergunta,
        5
    )

    distancias.extend(
        resultado["distances"][0]
    )

np.std(
    distancias
)

/home/spark-e9174953-5e22-41d1-af44-48/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 103MiB/s]


np.float64(0.1685201459597867)

In [0]:
collection.count()

48272

In [0]:
kb_pd["review_score"]

0        1
1        1
2        5
3        5
4        5
        ..
39759    1
39760    3
39761    2
39762    3
39763    1
Name: review_score, Length: 39764, dtype: object

In [0]:
# =====================================================
# AVALIAÇÃO DO RETRIEVER
# =====================================================
#
# Objetivo:
# Avaliar o desempenho do banco vetorial
# utilizando perguntas reais.
#
# Métricas:
#
# 1) Tempo médio de recuperação
# 2) Distância média
# 3) Desvio padrão das distâncias
# 4) Quantidade de documentos indexados
#
# =====================================================

import time
import numpy as np

# =====================================================
# MODELO UTILIZADO
# =====================================================

modelo = "paraphrase-multilingual-MiniLM-L12-v2"

# =====================================================
# LISTAS PARA ARMAZENAR AS MÉTRICAS
# =====================================================

tempos = []

distancias = []

# =====================================================
# EXECUTA AS CONSULTAS
# =====================================================

for pergunta in PERGUNTAS:

    inicio = time.time()

    resultado = buscar(
        collection,
        pergunta,
        5
    )

    fim = time.time()

    # Tempo da consulta
    tempos.append(
        fim - inicio
    )

    # Guarda as distâncias retornadas
    distancias.extend(
        resultado["distances"][0]
    )

# =====================================================
# MÉTRICA 1
# TEMPO MÉDIO
# =====================================================

tempo_medio = np.mean(
    tempos
)

# =====================================================
# MÉTRICA 2
# DISTÂNCIA MÉDIA
# =====================================================

distancia_media = np.mean(
    distancias
)

# =====================================================
# MÉTRICA 3
# DESVIO PADRÃO
# =====================================================

desvio = np.std(
    distancias
)

# =====================================================
# DASHBOARD DE MÉTRICAS
# =====================================================

metricas = {

    "documentos_indexados":
        collection.count(),

    "embedding_model":
        modelo,

    "dimensao_embedding":
        384,

    "tempo_medio_busca":
        round(
            tempo_medio,
            4
        ),

    "distancia_media":
        round(
            distancia_media,
            4
        ),

    "desvio_padrao":
        round(
            desvio,
            4
        ),

    "top_k":
        5

}

# =====================================================
# EXIBIÇÃO DAS MÉTRICAS
# =====================================================

print("\n")
print("=" * 60)
print("MÉTRICAS DO RETRIEVER")
print("=" * 60)

for k,v in metricas.items():

    print(
        f"{k}: {v}"
    )



MÉTRICAS DO RETRIEVER
documentos_indexados: 48272
embedding_model: paraphrase-multilingual-MiniLM-L12-v2
dimensao_embedding: 384
tempo_medio_busca: 0.2701
distancia_media: 3.2428
desvio_padrao: 0.1685
top_k: 5


In [0]:
# =====================================================
# CACHE DE CONSULTAS
# =====================================================

CACHE = {}

cache_hit = 0
cache_miss = 0

Cache de Perguntas

No seu caso é interessante porque  repetindo várias consultas durante a experimentação.

In [0]:
# =====================================================
# CACHE DE CONSULTAS
# =====================================================

CACHE = {}

cache_hit = 0
cache_miss = 0

Função de Busca com Cache

In [0]:
def buscar_com_cache(
        collection,
        pergunta,
        top_k=5
    ):

    global cache_hit
    global cache_miss

    chave = f"{pergunta}_{top_k}"

    # Já existe em memória
    if chave in CACHE:

        cache_hit += 1

        return CACHE[chave]

    # Consulta no banco vetorial
    resultado = collection.query(

        query_texts=[pergunta],

        n_results=top_k

    )

    CACHE[chave] = resultado

    cache_miss += 1

    return resultado

In [0]:
buscar_com_cache(
    collection,
    "atraso na entrega",
    5
)

buscar_com_cache(
    collection,
    "atraso na entrega",
    5
)

buscar_com_cache(
    collection,
    "atraso na entrega",
    5
)


{'ids': [['7192', '10274', '39925', '34179', '20844']],
 'embeddings': None,
 'documents': [['Categoria do Produto: informatica_acessorios Nota da Avaliação: 4 Sentimento: POSITIVO Comentário do Cliente: RECOMENDO A LOJA E O PRODUTO, SÓ NÃO É PEFEITO PORQUE OS CORREIOS NÃO ENTREGAM ONDE MORO POR CAUSA DOS ASSALTOS. MAS O RESTENTE É PERFEITO, ALOJA, O PRODUTO O ATENDIMENTO, O PRAZO. OBRIGADA, VOLTAREI',
   'Categoria do Produto: cama_mesa_banho Nota da Avaliação: 2 Sentimento: NEGATIVO Comentário do Cliente: Um verdadeiro absurdo hj já são 09 de agosto e nada da entrega do meu produto. Uma verdadeira falta de respeito. O stark deveria escolher melhor o seu meio de transpirte',
   'Categoria do Produto: ferramentas_jardim Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Continuo aguardando o produto e pelo rastreio está preso em Curitiba. Era p ter sido entregue dia 24 e hj dia 27 continua em outro estado.\n',
   'Categoria do Produto: relogios_presentes Nota da Avaliação

Métrica de Cache

In [0]:
print(
    f"Cache Hits  : {cache_hit}"
)

print(
    f"Cache Miss  : {cache_miss}"
)

taxa_cache = (
    cache_hit /
    (cache_hit + cache_miss)
) * 100

print(
    f"Cache Hit Rate: {taxa_cache:.2f}%"
)

Cache Hits  : 2
Cache Miss  : 1
Cache Hit Rate: 66.67%


Sem cache
Pergunta
↓
Embedding
↓
Banco Vetorial
↓
Retorno

Com cache
Pergunta já feita
↓
Busca em memória
↓
Resposta imediata

In [0]:
#Instalação LANCEDB
%pip install lancedb sentence-transformers

dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 80.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [lancedb]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


ETAPA 1 - Configuração

In [0]:
# =====================================================
# CONFIGURAÇÕES
# =====================================================

TOP_K = 5

MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

CAMINHO_LANCEDB = "/tmp/lancedb_olist"

ETAPA 2 - Ler Knowledge Base

Utilize a mesma Gold.

In [0]:
# =====================================================
# KNOWLEDGE BASE
# =====================================================

kb = spark.table(
    "projeto.gold."
    "customer_reviews_knowledge_base"
)

display(kb.limit(5))

document_id,review_id,order_id,review_score,sentimento,product_category_name,dias_entrega,document_text
0,72730619a00e0ecdecf0a4ff862d8996,af76a45a10524549102e010f2bd9d3cd,1,NEGATIVO,papelaria,14,Categoria do Produto: papelaria Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: comprei tres pacotes de cinco folhas cada de papel transfer para tecido escuro e so recebi dois
1,346317f9aff540395fe46893611854f8,546ffb713086001ce9f0009158f88b73,1,NEGATIVO,esporte_lazer,3,Categoria do Produto: esporte_lazer Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Não funciona não faz sincronismo
2,7914fa319f12631679d1704d7dcd17df,06ecfa8dfae87263e05d953aa561603d,5,POSITIVO,automotivo,6,Categoria do Produto: automotivo Nota da Avaliação: 5 Sentimento: POSITIVO Comentário do Cliente: Recomendo
3,49f6f0bbf53f6d961deefc4d2fb0a6cd,4b9d1bbbeed0481ecc124c66e355896f,5,POSITIVO,moveis_quarto,8,Categoria do Produto: moveis_quarto Nota da Avaliação: 5 Sentimento: POSITIVO Comentário do Cliente: Chegou antes do prazo é a cabeceira é linda.
4,1545c0d2c381662ba869e91e4c1aaca1,c589bcc1414f7d7d949f353b1d1a45d9,5,POSITIVO,esporte_lazer,56,Categoria do Produto: esporte_lazer Nota da Avaliação: 5 Sentimento: POSITIVO Comentário do Cliente: Até agora tudo bem estou esperando a entrega do produto


In [0]:
from pyspark.sql import functions as F

kb = (

    kb

    .filter(
        F.length(
            F.col("document_text")
        ) > 20
    )

    .dropDuplicates(
        ["document_text"]
    )

)

ETAPA 4 - Converter

In [0]:
kb_pd = kb.select(

    "document_id",

    "document_text",

    "review_score",

    "sentimento",

    "product_category_name",

    "dias_entrega"

).toPandas()

ETAPA 5 - Embeddings

In [0]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    MODELO
)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-e9174953-5e22-41d1-af44-487fc702505e/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
embeddings = model.encode(

    kb_pd["document_text"].tolist(),

    batch_size=128,

    show_progress_bar=True

)

Batches:   0%|          | 0/311 [00:00<?, ?it/s]

ETAPA 6 - Conectar LanceDB

In [0]:
import lancedb

db = lancedb.connect(
    CAMINHO_LANCEDB
)

ETAPA 7 - Estrutura da Tabela

In [0]:
dados = []

In [0]:
for i in range(
    len(kb_pd)
):

    dados.append(

        {

            "id":

            str(
                kb_pd.iloc[i]
                ["document_id"]
            ),

            "text":

            kb_pd.iloc[i]
            ["document_text"],

            "vector":

            embeddings[i],

            "score":

            int(
                kb_pd.iloc[i]
                ["review_score"]
            )

            if str(
                kb_pd.iloc[i]
                ["review_score"]
            ).strip().isdigit()

            else 0,

            "sentimento":

            str(
                kb_pd.iloc[i]
                ["sentimento"]
            ),

            "categoria":

            str(
                kb_pd.iloc[i]
                ["product_category_name"]
            )

        }

    )

ETAPA 8 - Criar Tabela

In [0]:
tbl = db.create_table(

    "olist_reviews",

    data=dados,

    mode="overwrite"

)

[2026-11-10T01:50:51Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/lancedb_olist/olist_reviews.lance, it will be created


ETAPA 9 - Busca Vetorial

In [0]:
def buscar_lancedb(

    pergunta,

    top_k=5

):

    vetor = model.encode(
        pergunta
    )

    retorno = (

        tbl.search(
            vetor
        )

        .limit(
            top_k
        )

        .to_pandas()

    )

    return retorno

ETAPA 10 - Teste
Python

In [0]:
resultado = buscar_lancedb(

    "atraso na entrega",

    top_k=5

)

display(resultado)

id text vector score sentimento categoria _distance 19655 Categoria do Produto: utilidades_domesticas Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Não é a primeira vez de atraso na entrega, quando consegue falar a atendente ainda pede um prazo de dois dias, fora o atraso, infelizmente as mudanças de entrega dos parceiros não está sendo pratica List(-0.014430261, 0.03180521, -0.123537034, 0.0037313674, -0.003044508, -0.038527675, -0.18065281, 0.027262164, 0.1515967, -0.094564535, -0.080235675, 0.22083795, 0.005958274, 0.055681076, -0.09364675, -0.057882093, 0.12584324, -0.23600678, 0.07074269, 0.06240681, 0.018527756, -0.08128401, 0.11039906, 0.18635277, 0.229591, -0.12643214, 0.08269721, -0.04446631, -0.0847918, -0.22088991, -0.08883953, 0.088874705, 0.06771226, -0.035021182, 0.14474972, 0.031339332, 0.0303002, -0.14531139, 0.020940915, -0.048848145, 0.06583309, -0.0845998, -0.08346022, 0.09247198, -0.1532284, 0.036117814, 0.041393824, -0.0037934002, -0.27704346, 0.15106983, -0.07103243, -0.02405462, -0.034298364, -0.0736194, -0.13959487, 0.017003572, 0.12835675, 0.07703076, -0.21440922, 0.14114796, 0.007941776, -0.17830704, -0.21942583, -0.01266029, -0.18114334, 0.023079818, -0.2756097, -0.2356789, -0.11636503, -0.018523952, -0.06997141, -0.0027607575, -0.2531406, -0.07933541, 0.014281533, 0.054659642, 0.024626847, 0.10533707, 0.22701296, -0.22965896, -0.14180966, -0.005701589, -0.13981634, -0.18130784, -0.1886399, -0.075739555, 0.042005524, 0.32736465, -0.0406982, -0.09731916, 0.252918, -0.004326995, 0.21233971, 0.11478615, -0.087766886, -0.10543707, -0.19061895, -0.043951783, -0.07333068, 0.24645023, 0.040657878, 0.3242561, -0.45868447, -0.02229541, -0.0164575, 0.1797392, 0.09301509, -0.0867087, -0.1161282, -0.010547075, -0.15952295, 0.1473061, -0.10469365, -0.20452198, -0.1838985, 0.37893772, 0.054471757, 0.2138555, 0.2803638, 0.13629577, -0.022336906, 0.010840096, 0.119306765, -0.20347868, -0.10085576, -0.14885324, 0.22050598, -0.13304135, -0.13498545, -0.101303816, 0.044722483, -0.012468363, -0.100782566, 0.050868012, 0.079110116, 0.010775152, 0.037973575, -0.025207816, 0.07337881, -0.036212385, 0.2619666, 0.05770144, -0.053165067, 0.12535585, 0.19511725, 0.29175594, 0.24824585, 0.022118289, -0.026392674, -0.017083352, -0.220302, 0.18859851, 0.13203347, -0.26326257, -0.19584039, 0.06745492, -0.022103745, -0.0012326895, -0.06555793, -0.10400128, -0.018428504, 0.030949438, -0.0016593243, -0.10401887, -0.049889226, -0.2807247, 0.15645406, -0.15362024, -0.004489882, 0.10926397, -0.07908373, -0.035351973, -0.267862, -0.15603247, 0.016951475, -0.03850004, 0.08033701, 0.037825547, -0.04917132, 8.1554515E-4, 0.18580052, 0.1541306, -0.16824129, -0.05910414, 0.18629116, -0.20558196, 0.0024592776, -0.13909273, 0.067945816, 0.04463739, -0.072897315, -0.27574715, 0.117061265, 0.24298795, -0.04391208, -0.005535469, 0.105680935, 0.04528562, 0.035971325, 0.17660742, -0.039263215, -0.005081907, 0.1295995, -0.014154582, 0.031575345, 0.016512094, 0.09938914, -0.22861923, 0.22609708, -0.002955906, -0.078812756, -0.040111694, 0.06608114, 0.052325483, 0.0017228911, 0.22737637, -0.18800746, 0.41683602, -0.16583133, 0.117739625, 0.01056987, 0.12673208, 0.41536424, -0.24510086, -0.057171028, 0.17193973, -0.37607566, -0.04989886, -0.014034151, 0.0066288537, 0.041654263, 0.038664337, 0.17591746, 0.19643845, -0.21359955, -0.1640899, 0.03703705, -0.04805691, -0.0356619, -0.0574794, -0.15679695, -0.049719207, 0.07823686, 0.0075092055, 0.113562785, 0.12919755, -0.1269984, -0.1771898, -0.036662694, 0.07466536, -0.084019005, -0.087308004, 0.014763268, -0.014075642, 0.045292426, -0.16725762, -0.17222932, 0.057551675, 0.16018659, -0.023636952, -0.044903137, 0.028563265, 0.17859326, 0.4575839, 0.26122123, 0.087479845, -0.008319259, -0.037261583, -0.072873846, -0.09950868, -0.06818886, -0.37371507, 0.15103213, 0.13165882, -0.021502672, 0.067272276, -0.018419826, 0.11441282, -0.017704478, 0.15916061, 0.0069450745, 0.103931814, 0.12492041,

ETAPA 11 - Cache

In [0]:
CACHE = {}
cache_hit = 0
cache_miss = 0

def buscar_com_cache(

    pergunta,

    top_k=5

):

    global cache_hit
    global cache_miss

    chave = f"{pergunta}_{top_k}"

    if chave in CACHE:

        cache_hit += 1

        return CACHE[chave]

    res = buscar_lancedb(
        pergunta,
        top_k
    )

    CACHE[chave] = res

    cache_miss += 1

    return res

ETAPA 12 - Avaliação
Python

In [0]:
PERGUNTAS = [

    "atraso na entrega",

    "produto quebrado",

    "problemas de transporte",

    "demora para chegar",

    "qualidade do produto"

]

ETAPA 13 - Dashboard

In [0]:
import time
import numpy as np

# Tempo medio de recuperacao por pergunta. Usa a busca com
# cache: a primeira execucao e cache miss, medindo assim o
# tempo real de recuperacao.

tempos = []

for pergunta in PERGUNTAS:

    inicio = time.time()

    buscar_com_cache(
        pergunta,
        5
    )

    fim = time.time()

    tempos.append(
        fim - inicio
    )

metricas = {

    "Banco":

    "LanceDB",

    "Modelo":

    MODELO,

    "Documentos":

    len(kb_pd),

    "Tempo Medio":

    round(
        np.mean(
            tempos
        ),
        4
    ),

    "Cache Hits":

    cache_hit,

    "Cache Miss":

    cache_miss,

    "Cache Hit Rate":

    round(

        cache_hit /
        (cache_hit + cache_miss)

        * 100,

        2

    )

}

In [0]:
for k,v in metricas.items():

    print(
        f"{k}: {v}"
    )

Banco: LanceDB
Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Documentos: 39764
Tempo Medio: 0.0
Cache Hits: 5
Cache Miss: 5
Cache Hit Rate: 50.0


Verificar modelos disponíveis


In [0]:
from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")


In [0]:
client.list_endpoints()

[{'name': 'databricks-gpt-oss-120b',
  'creation_timestamp': 1754179200000,
  'last_updated_timestamp': 1754179200000,
  'state': {'ready': 'READY', 'config_update': 'NOT_UPDATING'},
  'config': {'served_entities': [{'name': 'databricks-gpt-oss-120b',
     'entity_name': 'system.ai.gpt-oss-120b',
     'type': 'FOUNDATION_MODEL',
     'foundation_model': {'name': 'system.ai.gpt-oss-120b',
      'display_name': 'GPT OSS 120B',
      'docs': 'https://docs.databricks.com/machine-learning/foundation-models/supported-models.html#openai-gpt-120b',
      'description': "GPT OSS 120B is a state-of-the-art, reasoning model with chain-of-thought and adjustable reasoning effort levels built and trained by OpenAI. It is OpenAI's flagship open-weight model that features a 128K token context window. The model is built for high-quality reasoning tasks.",
      'price': '8.571',
      'input_price': '2.143',
      'price_unit': 'DBUs per 1M tokens',
      'pricing_model': 'Pay-per-token',
      'model_

In [0]:
response = client.predict(

    endpoint="databricks-meta-llama-3-1-8b-instruct",

    inputs={

        "messages":[

            {

                "role":"user",

                "content":"O que é RAG?"

            }

        ]

    }
)

print(
    response
)

{'id': 'chatcmpl_94eaaeb2-43d7-4e12-bf11-917a4b7552a6_90d45d40-f415-468f-98f9-803e93a676e7', 'object': 'chat.completion', 'created': 1794275959, 'model': 'meta-llama-3.1-8b-instruct-110524', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'RAG é um conhecido site da internet russo que era uma grande comunidade online de aplicativos como Jogos, aplicativos, afiliados, etc, em 2019 o site fecha.'}, 'finish_reason': 'stop', 'logprobs': None}], 'usage': {'prompt_tokens': 16, 'completion_tokens': 43, 'total_tokens': 59, 'cache_read_input_tokens': 0, 'prompt_tokens_details': {'cached_tokens': 0}}}


Função 1 - Recuperar Contexto USANDO O LLM DATABRICKS


In [0]:
def recuperar_contexto(
        pergunta,
        top_k=5
    ):

    vetor = model.encode(
        pergunta
    )

    resultados = (

        tbl

        .search(vetor)

        .limit(top_k)

        .to_pandas()

    )

    contexto = "\n\n".join(
        resultados["text"].tolist()
    )

    return contexto, resultados

Teste

In [0]:
contexto, docs = recuperar_contexto(

    "atraso na entrega",

    5

)

print(contexto)

Categoria do Produto: utilidades_domesticas Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Não é a primeira vez de atraso na entrega, quando consegue falar a atendente ainda pede um prazo de dois dias, fora o atraso, infelizmente as mudanças de entrega dos parceiros não está sendo pratica

Categoria do Produto: eletrodomesticos Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: A entrega está atrasada quase uma semana. No dia 11/09 Há dois registros FALSOS de entrega no histórico do site. Nos Correios consta que a encomenda não chegou ao destino.

Categoria do Produto: cama_mesa_banho Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Quero saber porque tanto atraso na minha encomenda, porque deram previsão de entrega para o dia 23/4 até hoje nada

Categoria do Produto: utilidades_domesticas Nota da Avaliação: 5 Sentimento: POSITIVO Comentário do Cliente: "A entrega atrasou em apenas um dia. Devido à despachos aquém do horário para distribu

Função 2 - RAG Completo

In [0]:
from mlflow.deployments import get_deploy_client

client = get_deploy_client(
    "databricks"
)


Primeira Pergunta Real

In [0]:
def perguntar_rag(

        pergunta,

        top_k=5

):

    contexto, evidencias = recuperar_contexto(

        pergunta,

        top_k

    )

    prompt = f"""

Você é um analista de experiência do cliente da Olist.

RESPONDA UTILIZANDO SOMENTE O CONTEXTO FORNECIDO.

Caso a informação não esteja presente,
informe que não encontrou evidências suficientes.

====================

CONTEXTO

{contexto}

====================

PERGUNTA:

{pergunta}

"""

    resposta = client.predict(

        endpoint="databricks-meta-llama-3-1-8b-instruct",

        inputs={

            "messages":[

                {

                    "role":"user",

                    "content":prompt

                }

            ]

        }

    )

    texto = (

        resposta["choices"][0]

        ["message"]

        ["content"]

    )

    return texto, evidencias

In [0]:
resposta, docs = perguntar_rag(

    "Quais são os principais problemas relatados pelos clientes?"

)

In [0]:
print(resposta)

Não encontrou evidências suficientes sobre os principais problemas relatados pelos clientes no contexto fornecido. Porém,
As principais observações apresentadas são:

- Pontualidade nos encaminhamentos e entregas 
- Solução caso de ocorrência 
- Previsibilidade dos prazos de entrega.


In [0]:
display(

    docs[

        [

            "text",

            "score",

            "sentimento",

            "categoria"

        ]

    ]

)

text,score,sentimento,categoria
"Categoria do Produto: utilidades_domesticas Nota da Avaliação: 5 Sentimento: POSITIVO Comentário do Cliente: O problema não é a loja, mas sim os Correios, que estão demorando demasiadamente para entregar os produtos.",5,POSITIVO,utilidades_domesticas
Categoria do Produto: informatica_acessorios Nota da Avaliação: 5 Sentimento: POSITIVO Comentário do Cliente: Minha encomenda deu um probleminha e com a ajuda do vendedor que fora do horário comercial me ajudou a resolver,5,POSITIVO,informatica_acessorios
Categoria do Produto: informatica_acessorios Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Já enviei email informando que está tendo problema sério na entrega do produto. Fui na agência dos Correios e eles me disseram que a responsabilidade é das lannister. Não chegou no destino.,1,NEGATIVO,informatica_acessorios
"Categoria do Produto: moveis_escritorio Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Produto mais que passou do prazo de entregar, tento resolver o problema com a empresa e não consigo falar com ninguém. Tudo que fazem é me passar para reclamar num sistema que não funciona.",1,NEGATIVO,moveis_escritorio
"Categoria do Produto: telefonia Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Além de receber o produto errado, não consegui efetuar a troca, devido a ineficiência do sistema, que não oferece ajuda suficiente.",1,NEGATIVO,telefonia


In [0]:
perguntas_demo = [

"Quais são os principais problemas relatados pelos clientes?",

"Por que os clientes dão nota 1?",

"Quais elogios aparecem com mais frequência?",

"Quais problemas estão relacionados à entrega?",

"O que clientes satisfeitos costumam mencionar?",

"Existe relação entre atraso e avaliações negativas?"

]

In [0]:
import numpy as np

# Tempo medio de recuperacao e taxa de acerto do cache,
# calculados a partir dos tempos e contadores da avaliacao

tempo_medio = round(

    np.mean(
        tempos
    ),

    4

)

taxa_cache = round(

    cache_hit /
    (cache_hit + cache_miss)

    * 100,

    2

)

rag_metricas = {

    "VectorDB":
    "LanceDB",

    "Embedding":
    "paraphrase-multilingual-MiniLM-L12-v2",

    "LLM":
    "Meta Llama 3.1 8B",

    "Documentos":
    tbl.count_rows(),

    "Top_K":
    5,

    "Tempo_Retriever":
    tempo_medio,

    "Cache_Hit_Rate":
    taxa_cache

}

imprimir as métricas do projeto

In [0]:
# =====================================================
# DASHBOARD FINAL DO RAG
# =====================================================

print("\n")
print("=" * 60)
print("MÉTRICAS FINAIS DO RAG")
print("=" * 60)

for k, v in rag_metricas.items():

    print(
        f"{k}: {v}"
    )



MÉTRICAS FINAIS DO RAG
VectorDB: LanceDB
Embedding: paraphrase-multilingual-MiniLM-L12-v2
LLM: Meta Llama 3.1 8B
Documentos: 39764
Top_K: 5
Tempo_Retriever: 0.0
Cache_Hit_Rate: 50.0


In [0]:
def recuperar_contexto_negativo(
        pergunta,
        top_k=10
):

    vetor = model.encode(
        pergunta
    )

    resultados = (

        tbl

        .search(vetor)

        .where(
            "sentimento = 'NEGATIVO'"
        )

        .limit(top_k)

        .to_pandas()

    )

    return resultados


In [0]:
docs = recuperar_contexto_negativo(

    "principais problemas dos clientes"

)


In [0]:
display(
    docs[
        [
            "sentimento",
            "score",
            "text"
        ]
    ]
)

sentimento,score,text
NEGATIVO,1,Categoria do Produto: climatizacao Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Atrasos acontecem nenhuma empresa está livre de passar por algum problema de logística. Mas o fato de não comunicarem seu cliente final é vergonhoso. Até o Mercado Livre tem controle sobre atrasos!
NEGATIVO,1,"Categoria do Produto: brinquedos Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: péssimo serviço de atendimento para resolver problemas, venho passando por um via crúcis para receber uma informação. e agora para cancelar a venda"
NEGATIVO,1,"Categoria do Produto: papelaria Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Sempre que alguma loja envia meus produtos pelos Correios, tenho problemas. Sempre atrasa muito, quando não extravia..."
NEGATIVO,1,"Categoria do Produto: cama_mesa_banho Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Depois de receber o produto avariado e solicitar a troca,começou a enrolação.Tamanha falta de comunicação entre a loja e o fornecedor do produto. Em NENHUM momento houve convergência de informações."
NEGATIVO,1,Categoria do Produto: moveis_decoracao Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: pessimo atendimento mercadoria chegandoatrazada vou fazer uma propanda pra nimguem mais comprar ai
NEGATIVO,2,"Categoria do Produto: moveis_escritorio Nota da Avaliação: 2 Sentimento: NEGATIVO Comentário do Cliente: Apesar dos produtos da loja serem de bom custo, o tempo da entrega e a falta de informações para o cliente sobre o andamento do pedido, deixa muito a desejar. Infelizmente não recomendo a terceiros."
NEGATIVO,1,"Categoria do Produto: moveis_decoracao Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Sou cliente há anos, porém sempre há divergências e problemas na entrega, consta como produto entregue, porém foi entregue apenas 1 produto, aguardo solução!"
NEGATIVO,1,"Categoria do Produto: informatica_acessorios Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Já faz algum tempo que a compra ocorreu e ainda não recebi a mercadoria, trata-se de um descaso com o cliente, já havia realizado outras compras pela stark, mas essa atitude inviabiliza novas aqui"
NEGATIVO,1,Categoria do Produto: cama_mesa_banho Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Se realmente tivenssem preocupação com os clientes nao deixariam a desejar em questao as entrega pois como que outros pedidos que foram feito no mesmo lugar e dia chegaram e eles colocam dificuldades.
NEGATIVO,1,"Categoria do Produto: bebes Nota da Avaliação: 1 Sentimento: NEGATIVO Comentário do Cliente: Preço do frente abusivo, prazo de entrega demorado e confuso, disseram que era pra eu cancelar a compra por problemas na entrega e dias depois entregaram o produto."


In [0]:
prompt = f"""

Você é um analista de experiência do cliente.

Analise os comentários abaixo.

Identifique:

1. Problemas mais frequentes.
2. Possíveis causas.
3. Padrões observados.

Baseie-se SOMENTE nas avaliações.

AVALIAÇÕES:

{contexto}

"""

In [0]:
positivos = len(
    docs[
        docs.sentimento == "POSITIVO"
    ]
)

negativos = len(
    docs[
        docs.sentimento == "NEGATIVO"
    ]
)

In [0]:
print(
    f"Positivos: {positivos}"
)

print(
    f"Negativos: {negativos}"
)

Positivos: 0
Negativos: 10


In [0]:
print("\n")
print("=" * 60)
print("MÉTRICAS FINAIS DO RAG")
print("=" * 60)

for chave, valor in rag_metricas.items():

    print(
        f"{chave:<25}: {valor}"
    )



MÉTRICAS FINAIS DO RAG
VectorDB                 : LanceDB
Embedding                : paraphrase-multilingual-MiniLM-L12-v2
LLM                      : Meta Llama 3.1 8B
Documentos               : 39764
Top_K                    : 5
Tempo_Retriever          : 0.0
Cache_Hit_Rate           : 50.0
